# 03 - Clean `rag_chunks_vi.jsonl` cho RAG medical chatbot

Notebook này làm sạch file `data/rag_processed/rag_chunks_vi.jsonl` và tạo file mới `data/rag_processed/clean_rag_chunks_vi.jsonl`.

## Lý thuyết các bước clean data cho RAG y khoa

1. **Không sửa dữ liệu gốc**: luôn đọc file raw và ghi ra file clean mới để có thể audit, so sánh và rollback.
2. **Validate JSONL/schema**: mỗi dòng phải parse được JSON, có các trường tối thiểu như `id`, `source`, `url`, `title`, `section`, `entity`, `content`.
3. **Sửa lỗi encoding/mojibake**: dữ liệu tiếng Việt bị lỗi kiểu `cÃ³ thá»ƒ`, `Ä‘Æ°á»£c` sẽ làm embedding rất kém. Cần thử decode lại và chỉ nhận bản sửa nếu score tiếng Việt tốt hơn.
4. **Loại boilerplate từ crawl**: header, menu, footer, chính sách, bản quyền, social link, recent activity, NDC packaging thường không giúp trả lời y khoa và gây nhiễu retrieval.
5. **Chuẩn hóa text**: bỏ markdown link thừa, HTML entity, ký tự lặp, khoảng trắng thừa; giữ lại thuật ngữ thuốc, liều, số đo, cảnh báo y khoa.
6. **Lọc chunk chất lượng thấp**: bỏ chunk quá ngắn, toàn link/menu, hoặc không còn đủ nội dung y khoa sau khi clean.
7. **Re-chunk cho RAG**: chunk nên vừa đủ nhỏ để retrieval chính xác, có overlap nhẹ để không mất ngữ cảnh.
8. **Giữ metadata và thêm audit fields**: giữ nguồn/URL/section/entity; thêm `cleaning_notes`, `original_id`, `chunk_index` để truy vết.
9. **Báo cáo chất lượng**: sau khi clean cần thống kê số dòng vào/ra, lỗi encoding còn lại, độ dài chunk, số nguồn/entity để biết dữ liệu có dùng được chưa.

> Lưu ý y khoa: dataset đã clean vẫn chỉ nên dùng làm knowledge base tham khảo. Chatbot cần guardrail: không tự chẩn đoán, không thay bác sĩ, ưu tiên khuyến cáo đi cấp cứu khi có red flags.

## Cell 1 - Khai báo đường dẫn và cấu hình

Cell này tìm project root, khai báo file input/output và các ngưỡng dùng khi clean/re-chunk.

In [1]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "rag_processed").exists():
            return path
    raise FileNotFoundError("Không tìm thấy thư mục data/rag_processed. Hãy chạy notebook trong repo Vietnamese-Medical-Chatbot.")

ROOT = find_project_root(Path.cwd().resolve())
INPUT_PATH = ROOT / "data" / "rag_processed" / "rag_chunks_vi.jsonl"
OUTPUT_PATH = ROOT / "data" / "rag_processed" / "clean_rag_chunks_vi.jsonl"
REPORT_PATH = ROOT / "data" / "rag_processed" / "clean_rag_chunks_vi_report.json"

# Ngưỡng có thể chỉnh tùy model embedding/generation bạn dùng.
MIN_CHARS = 300
TARGET_CHARS = 1800
MAX_CHARS = 2600
OVERLAP_CHARS = 250

print("ROOT:", ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("REPORT_PATH:", REPORT_PATH)


ROOT: D:\NLP\Vietnamese-Medical-Chatbot
INPUT_PATH: D:\NLP\Vietnamese-Medical-Chatbot\data\rag_processed\rag_chunks_vi.jsonl
OUTPUT_PATH: D:\NLP\Vietnamese-Medical-Chatbot\data\rag_processed\clean_rag_chunks_vi.jsonl
REPORT_PATH: D:\NLP\Vietnamese-Medical-Chatbot\data\rag_processed\clean_rag_chunks_vi_report.json


## Cell 2 - Import thư viện và định nghĩa bộ đo chất lượng

Cell này dùng thư viện chuẩn của Python để đọc JSONL, đo lỗi mojibake, đo ký tự tiếng Việt, và thống kê độ dài.

In [2]:
import html
import json
import re
import statistics
from collections import Counter, defaultdict
from copy import deepcopy

MOJIBAKE_RE = re.compile(r"(?:Ã|Â|Ä|Æ|Ð|ð|â|áº|á»|\ufffd)")
VI_CHAR_RE = re.compile(r"[ăâêôơưđĂÂÊÔƠƯĐàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵÀÁẢÃẠẰẮẲẴẶẦẤẨẪẬÈÉẺẼẸỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌỒỐỔỖỘỜỚỞỠỢÙÚỦŨỤỪỨỬỮỰỲÝỶỸỴ]")
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MARKDOWN_LINK_RE = re.compile(r"\[([^\]]+)\]\(([^)]+)\)")
EMPTY_MARKDOWN_LINK_RE = re.compile(r"\[\s*\]\([^)]*\)")

REQUIRED_FIELDS = [
    "id", "source", "url", "title", "section", "entity", "entity_type",
    "document_type", "topic_group", "patient_population", "content",
]

def mojibake_count(text: str) -> int:
    return len(MOJIBAKE_RE.findall(text or ""))

def vi_char_count(text: str) -> int:
    return len(VI_CHAR_RE.findall(text or ""))

def quality_score(text: str) -> float:
    text = text or ""
    return vi_char_count(text) * 2.0 - mojibake_count(text) * 8.0 - text.count("�") * 20.0

def percentile(values, q):
    if not values:
        return 0
    values = sorted(values)
    idx = int((len(values) - 1) * q)
    return values[idx]


## Cell 3 - Đọc JSONL và kiểm tra schema

Cell này parse từng dòng JSONL, báo lỗi nếu có dòng JSON hỏng, và kiểm tra các field bắt buộc.

In [3]:
items = []
invalid_lines = []

with INPUT_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            items.append(json.loads(line))
        except json.JSONDecodeError as exc:
            invalid_lines.append({"line_no": line_no, "error": str(exc), "line_head": line[:200]})

missing_fields = Counter()
for item in items:
    for field in REQUIRED_FIELDS:
        if field not in item:
            missing_fields[field] += 1

print("Tổng record đọc được:", len(items))
print("Dòng JSON lỗi:", len(invalid_lines))
print("Field thiếu:", dict(missing_fields))

if invalid_lines[:3]:
    print("Ví dụ dòng lỗi:")
    for row in invalid_lines[:3]:
        print(row)


Tổng record đọc được: 189
Dòng JSON lỗi: 0
Field thiếu: {}


## Cell 4 - Thống kê chất lượng trước khi clean

Cell này đo số chunk có lỗi encoding, nhiễu crawl/link, phân bố nguồn, entity và độ dài nội dung.

In [4]:
def dataset_stats(rows):
    lengths = [len(str(row.get("content", ""))) for row in rows]
    return {
        "records": len(rows),
        "unique_sources": len({row.get("source") for row in rows}),
        "unique_urls": len({row.get("url") for row in rows}),
        "unique_entities": len({row.get("entity") for row in rows}),
        "mojibake_records": sum(1 for row in rows if mojibake_count(str(row.get("content", ""))) > 0),
        "link_records": sum(1 for row in rows if re.search(r"https?://|\]\(", str(row.get("content", "")))),
        "min_len": min(lengths) if lengths else 0,
        "p25_len": percentile(lengths, 0.25),
        "median_len": percentile(lengths, 0.50),
        "p75_len": percentile(lengths, 0.75),
        "p90_len": percentile(lengths, 0.90),
        "max_len": max(lengths) if lengths else 0,
        "sources": dict(Counter(row.get("source", "unknown") for row in rows).most_common()),
        "document_types": dict(Counter(row.get("document_type", "unknown") for row in rows).most_common()),
        "topic_groups": dict(Counter(row.get("topic_group", "unknown") for row in rows).most_common()),
    }

raw_stats = dataset_stats(items)
print(json.dumps(raw_stats, ensure_ascii=False, indent=2))


{
  "records": 189,
  "unique_sources": 6,
  "unique_urls": 50,
  "unique_entities": 40,
  "mojibake_records": 182,
  "link_records": 168,
  "min_len": 1271,
  "p25_len": 3819,
  "median_len": 4355,
  "p75_len": 4848,
  "p90_len": 5103,
  "max_len": 5904,
  "sources": {
    "MedlinePlus": 64,
    "MedlinePlus Drug Information": 39,
    "NCBI Bookshelf LactMed": 37,
    "DailyMed": 28,
    "NIDDK": 14,
    "NHS": 7
  },
  "document_types": {
    "drug_information": 39,
    "pregnancy_lactation_reference": 37,
    "symptom_triage": 36,
    "disease_overview": 34,
    "special_population_reference": 34,
    "overdose_toxicology": 9
  },
  "topic_groups": {
    "Overdose & Triage": 45,
    "Drug Safety": 39,
    "Pregnancy & Lactation": 37,
    "Disease Knowledge": 34,
    "Pediatric & Special Populations": 34
  }
}


## Cell 5 - Hàm sửa lỗi encoding/mojibake

Cell này thử nhiều cách decode lại chuỗi bị hỏng. Notebook chỉ nhận bản sửa nếu điểm chất lượng tiếng Việt tốt hơn bản gốc.

In [5]:
def try_redecode(text: str, source_encoding: str) -> str:
    try:
        return text.encode(source_encoding, errors="strict").decode("utf-8", errors="strict")
    except UnicodeError:
        return text

def is_legacy_byte_char(ch: str) -> bool:
    """True nếu ký tự có thể là byte cũ bị decode nhầm thành Unicode."""
    if ord(ch) <= 255:
        return True
    try:
        ch.encode("cp1252")
        return True
    except UnicodeError:
        return False

def repair_legacy_run(text: str) -> str:
    candidates = [text]
    for enc in ["cp1252", "latin1"]:
        candidates.append(try_redecode(text, enc))
    return max(candidates, key=quality_score)

def repair_mixed_mojibake(text: str) -> str:
    # Sửa theo từng đoạn byte-like để không phá phần tiếng Việt vốn đã đúng.
    parts = []
    buf = []

    def flush():
        if not buf:
            return
        segment = "".join(buf)
        if mojibake_count(segment) > 0:
            parts.append(repair_legacy_run(segment))
        else:
            parts.append(segment)
        buf.clear()

    for ch in text:
        if is_legacy_byte_char(ch):
            buf.append(ch)
        else:
            flush()
            parts.append(ch)
    flush()
    return "".join(parts)

def fix_mojibake_once(text: str) -> str:
    candidates = [text, repair_mixed_mojibake(text)]
    for enc in ["latin1", "cp1252"]:
        candidates.append(try_redecode(text, enc))

    # Một số chuỗi bị hỏng hai lần, thử thêm vòng decode thứ hai.
    for candidate in list(candidates):
        for enc in ["latin1", "cp1252"]:
            candidates.append(try_redecode(candidate, enc))

    best = max(candidates, key=quality_score)
    return best

def fix_mojibake(text: str) -> tuple[str, bool]:
    if not isinstance(text, str):
        return "", False
    before = text
    after = fix_mojibake_once(before)
    improved = quality_score(after) > quality_score(before) and mojibake_count(after) <= mojibake_count(before)
    return (after if improved else before), improved

for sample in items[:3]:
    fixed, changed = fix_mojibake(sample.get("content", ""))
    print("ID:", sample.get("id"), "changed=", changed)
    print("Before:", sample.get("content", "")[:180])
    print("After :", fixed[:180])
    print("-" * 80)


ID: drug-safety__warfarin__735e158fbd__001 changed= False
Before: Trang web chính thức của chính phủ Hoa Kỳ Đây là cách bạn biết **Trang web chính thức sử dụng .gov** Một trang web có đuôi .gov thuộc về một tổ chức chính phủ chính thức tại Hoa Kỳ
After : Trang web chính thức của chính phủ Hoa Kỳ Đây là cách bạn biết **Trang web chính thức sử dụng .gov** Một trang web có đuôi .gov thuộc về một tổ chức chính phủ chính thức tại Hoa Kỳ
--------------------------------------------------------------------------------
ID: drug-safety__warfarin__735e158fbd__002 changed= False
Before: có thể bị cắt hoặc chấn thương. Tránh các hoạt động hoặc môn thể thao có nguy cơ bị chấn thương. Gọi cho bác sĩ nếu bạn có tình trạng chảy máu bất thường hoặc nếu bạn ngã và bị thư
After : có thể bị cắt hoặc chấn thương. Tránh các hoạt động hoặc môn thể thao có nguy cơ bị chấn thương. Gọi cho bác sĩ nếu bạn có tình trạng chảy máu bất thường hoặc nếu bạn ngã và bị thư
-----------------------------------------------

## Cell 6 - Hàm loại boilerplate và chuẩn hóa nội dung

Cell này bỏ các đoạn thường là header/footer/menu từ MedlinePlus, NCBI, NHS, DailyMed; sau đó chuẩn hóa markdown link, URL và khoảng trắng.

In [6]:
DROP_LINE_PATTERNS = [
    r"^Trang web chính thức của chính phủ Hoa Kỳ",
    r"^Một trang web chính thức của chính phủ Hoa Kỳ",
    r"^Đây là cách bạn biết",
    r"^Trang web \.gov",
    r"^Menu\b",
    r"^Hiển thị Tìm kiếm",
    r"^Bạn đang ở đây:",
    r"^URL của trang này:",
    r"^Trang này có hữu ích không",
    r"^CóKhông",
    r"^Cảm ơn phản hồi",
    r"^Bản tuyên bố miễn trừ trách nhiệm",
    r"^AHFS",
    r"^©",
    r"^Bản quyền",
    r"^Hỗ trợ khách hàng",
    r"^Kết nối với",
    r"^Chính sách Web",
    r"^Nguyên tắc liên kết",
    r"^Phần mềm xem",
    r"^Đối với nhà phát triển",
    r"^Thư viện Quốc gia",
    r"^8600 Rockville Pike",
    r"^Recent Activity$",
    r"^Hoạt động duyệt web",
    r"^Ghi lại hoạt động",
    r"^Bật lại ghi lại",
    r"^Theo dõi NCBI",
    r"^Chia sẻ trên",
    r"^Số lần xem",
    r"^Các lần xem",
    r"^PubReader$",
    r"^Trích dẫn trang này",
    r"^Phiên bản PDF",
    r"^Xem Tất cả Các Phần",
    r"^Đóng Tất cả Các Phần",
    r"^Thông tin tiếp thị$",
    r"^Thông tin quảng bá$",
    r"^Nhà sản xuất",
    r"^NDC[:\s]",
    r"^Mã số hàng hóa",
    r"^Mã số mặt hàng",
    r"^Tên cơ sở",
    r"^Đăng nhập",
    r"^Tuyển dụng$",
]
DROP_LINE_RE = re.compile("|".join(f"(?:{p})" for p in DROP_LINE_PATTERNS), flags=re.IGNORECASE)

CUT_AFTER_PATTERNS = [
    r"## Trang này có hữu ích không",
    r"### Recent Activity",
    r"Hoạt động duyệt web của bạn",
    r"\[Xem Tất cả Các Phần\]",
    r"### Các lần xem",
    r"### Số lần xem",
    r"## Liên kết hỗ trợ",
]
CUT_AFTER_RE = re.compile("|".join(CUT_AFTER_PATTERNS), flags=re.IGNORECASE)

def strip_after_noise(text: str) -> str:
    match = CUT_AFTER_RE.search(text)
    if match:
        return text[:match.start()].strip()
    return text

def remove_markdown_links(text: str) -> str:
    text = EMPTY_MARKDOWN_LINK_RE.sub(" ", text)
    text = MARKDOWN_LINK_RE.sub(lambda m: m.group(1).strip(), text)
    text = URL_RE.sub(" ", text)
    return text

def normalize_text(text: str) -> str:
    text = html.unescape(text or "")
    text = text.replace("\u00a0", " ")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([.!?])(?=[A-ZÀ-Ỵ])", r"\1 ", text)
    text = re.sub(r"\*{3,}", "**", text)
    return text.strip()

def remove_boilerplate(text: str) -> tuple[str, list[str]]:
    notes = []
    before = text
    text = strip_after_noise(text)
    if text != before:
        notes.append("cut_footer_or_navigation_tail")

    text = remove_markdown_links(text)
    lines = []
    dropped = 0
    for raw_line in text.splitlines():
        line = raw_line.strip(" -•\t")
        if not line:
            continue
        if DROP_LINE_RE.search(line):
            dropped += 1
            continue
        # Bỏ dòng gần như chỉ là link/navigation ngắn.
        if len(line) < 90 and re.search(r"^(Trang chủ|Chủ đề sức khỏe|Thuốc và Bổ sung|Di truyền|Từ điển y học|Về MedlinePlus)$", line, re.I):
            dropped += 1
            continue
        lines.append(line)

    if dropped:
        notes.append(f"dropped_boilerplate_lines:{dropped}")
    return normalize_text("\n".join(lines)), notes


## Cell 7 - Clean từng record

Cell này áp dụng sửa encoding, bỏ boilerplate, chuẩn hóa text và gắn ghi chú audit vào từng record.

In [7]:
cleaned_base = []
drop_reasons = Counter()
note_counter = Counter()

for item in items:
    row = deepcopy(item)
    original_content = str(row.get("content", ""))
    notes = []

    fixed_content, changed_encoding = fix_mojibake(original_content)
    if changed_encoding:
        notes.append("fixed_mojibake")

    clean_content, boilerplate_notes = remove_boilerplate(fixed_content)
    notes.extend(boilerplate_notes)

    clean_content = normalize_text(clean_content)
    content_len = len(clean_content)

    if content_len < MIN_CHARS:
        drop_reasons["too_short_after_clean"] += 1
        continue
    if mojibake_count(clean_content) > max(5, content_len * 0.005):
        drop_reasons["still_too_much_mojibake"] += 1
        continue
    if len(URL_RE.findall(clean_content)) > 8:
        drop_reasons["too_many_urls"] += 1
        continue

    row["content"] = clean_content
    row["original_id"] = row.get("id")
    row["cleaning_notes"] = notes
    row["original_content_chars"] = len(original_content)
    row["clean_content_chars"] = content_len
    cleaned_base.append(row)
    note_counter.update(notes)

print("Record sau clean bước đầu:", len(cleaned_base), "/", len(items))
print("Lý do bị loại:", dict(drop_reasons))
print("Ghi chú clean phổ biến:", dict(note_counter.most_common(10)))

for row in cleaned_base[:3]:
    print("\nID:", row["id"])
    print("Notes:", row["cleaning_notes"])
    print(row["content"][:500])


Record sau clean bước đầu: 131 / 189
Lý do bị loại: {'too_short_after_clean': 33, 'still_too_much_mojibake': 25}
Ghi chú clean phổ biến: {'cut_footer_or_navigation_tail': 25}

ID: drug-safety__warfarin__735e158fbd__002
Notes: []
có thể bị cắt hoặc chấn thương. Tránh các hoạt động hoặc môn thể thao có nguy cơ bị chấn thương. Gọi cho bác sĩ nếu bạn có tình trạng chảy máu bất thường hoặc nếu bạn ngã và bị thương, đặc biệt là nếu bạn va đầu. Hãy giữ lịch hẹn với bác sĩ và phòng thí nghiệm. Bác sĩ của bạn sẽ chỉ định xét nghiệm máu (PT [test prothrombin] được báo cáo dưới dạng giá trị INR [international normalized ratio]) định kỳ để kiểm tra phản ứng của cơ thể với warfarin. Nếu bác sĩ bảo bạn ngừng dùng warfarin, tác dụng c

ID: drug-safety__warfarin__735e158fbd__003
Notes: []
bệnh lý đường tiêu hóa như tiêu chảy, hoặc bệnh sprue (một phản ứng dị ứng với các loại hạt gây tiêu chảy), hoặc ống thông tiểu (một ống nhựa mềm được đặt vào bàng quang để giúp dẫn lưu nước tiểu ra ngoài). * Hãy thô

## Cell 8 - Re-chunk nội dung cho RAG

Cell này chia lại các content dài thành chunk nhỏ hơn, ưu tiên ngắt ở heading/câu/khoảng trắng và có overlap nhẹ.

In [8]:
def choose_split_point(text: str, start: int, max_end: int, min_end: int) -> int:
    window = text[start:max_end]
    candidates = []
    for pattern in [r"\n## ", r"\n### ", r"\. ", r"\? ", r"! ", r"; ", r"\n", r", "]:
        matches = list(re.finditer(pattern, window))
        if matches:
            pos = start + matches[-1].end()
            if pos >= min_end:
                candidates.append(pos)
    if candidates:
        return max(candidates)
    return max_end

def split_text_for_rag(text: str) -> list[str]:
    text = normalize_text(text)
    if len(text) <= MAX_CHARS:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        remaining = len(text) - start
        if remaining <= MAX_CHARS:
            tail = text[start:].strip()
            if tail:
                chunks.append(tail)
            break

        max_end = min(len(text), start + MAX_CHARS)
        min_end = min(len(text), start + TARGET_CHARS)
        end = choose_split_point(text, start, max_end, min_end)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = max(end - OVERLAP_CHARS, start + 1)

    # Gộp tail quá ngắn vào chunk trước nếu có thể.
    if len(chunks) >= 2 and len(chunks[-1]) < MIN_CHARS:
        chunks[-2] = normalize_text(chunks[-2] + "\n" + chunks[-1])
        chunks.pop()
    return chunks

final_rows = []
for row in cleaned_base:
    parts = split_text_for_rag(row["content"])
    total_parts = len(parts)
    for idx, part in enumerate(parts, start=1):
        if len(part) < MIN_CHARS:
            drop_reasons["too_short_after_rechunk"] += 1
            continue
        out = deepcopy(row)
        out["content"] = part
        out["chunk_index"] = idx
        out["chunk_count"] = total_parts
        out["id"] = f"{row.get('id', 'chunk')}__clean_{idx:03d}"
        out["clean_content_chars"] = len(part)
        final_rows.append(out)

print("Record cuối sau re-chunk:", len(final_rows))
print("Lý do bị loại cập nhật:", dict(drop_reasons))


Record cuối sau re-chunk: 218
Lý do bị loại cập nhật: {'too_short_after_clean': 33, 'still_too_much_mojibake': 25}


## Cell 9 - Kiểm tra chất lượng sau clean

Cell này in thống kê sau clean để so sánh với dữ liệu ban đầu trước khi ghi file.

In [9]:
clean_stats = dataset_stats(final_rows)
remaining_mojibake_examples = [
    {"id": row.get("id"), "head": row.get("content", "")[:250], "mojibake_count": mojibake_count(row.get("content", ""))}
    for row in final_rows
    if mojibake_count(row.get("content", "")) > 0
][:5]

print("=== RAW STATS ===")
print(json.dumps(raw_stats, ensure_ascii=False, indent=2))
print("\n=== CLEAN STATS ===")
print(json.dumps(clean_stats, ensure_ascii=False, indent=2))
print("\nVí dụ còn nghi mojibake:")
print(json.dumps(remaining_mojibake_examples, ensure_ascii=False, indent=2))


=== RAW STATS ===
{
  "records": 189,
  "unique_sources": 6,
  "unique_urls": 50,
  "unique_entities": 40,
  "mojibake_records": 182,
  "link_records": 168,
  "min_len": 1271,
  "p25_len": 3819,
  "median_len": 4355,
  "p75_len": 4848,
  "p90_len": 5103,
  "max_len": 5904,
  "sources": {
    "MedlinePlus": 64,
    "MedlinePlus Drug Information": 39,
    "NCBI Bookshelf LactMed": 37,
    "DailyMed": 28,
    "NIDDK": 14,
    "NHS": 7
  },
  "document_types": {
    "drug_information": 39,
    "pregnancy_lactation_reference": 37,
    "symptom_triage": 36,
    "disease_overview": 34,
    "special_population_reference": 34,
    "overdose_toxicology": 9
  },
  "topic_groups": {
    "Overdose & Triage": 45,
    "Drug Safety": 39,
    "Pregnancy & Lactation": 37,
    "Disease Knowledge": 34,
    "Pediatric & Special Populations": 34
  }
}

=== CLEAN STATS ===
{
  "records": 218,
  "unique_sources": 6,
  "unique_urls": 43,
  "unique_entities": 34,
  "mojibake_records": 194,
  "link_records": 21,

## Cell 10 - Ghi file clean vào `data/rag_processed`

Cell này ghi `clean_rag_chunks_vi.jsonl` và một file report nhỏ để audit lại quá trình clean.

In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8", newline="\n") as f:
    for row in final_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

report = {
    "input_path": str(INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "raw_stats": raw_stats,
    "clean_stats": clean_stats,
    "drop_reasons": dict(drop_reasons),
    "cleaning_note_counts": dict(note_counter.most_common()),
    "config": {
        "MIN_CHARS": MIN_CHARS,
        "TARGET_CHARS": TARGET_CHARS,
        "MAX_CHARS": MAX_CHARS,
        "OVERLAP_CHARS": OVERLAP_CHARS,
    },
}

with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Done!")
print("Saved clean JSONL:", OUTPUT_PATH)
print("Saved report:", REPORT_PATH)
print("Final records:", len(final_rows))


Done!
Saved clean JSONL: D:\NLP\Vietnamese-Medical-Chatbot\data\rag_processed\clean_rag_chunks_vi.jsonl
Saved report: D:\NLP\Vietnamese-Medical-Chatbot\data\rag_processed\clean_rag_chunks_vi_report.json
Final records: 218


## Cell 11 - Đọc lại file output để kiểm tra nhanh

Cell này mở lại file vừa ghi, parse JSONL và in vài dòng mẫu để chắc chắn file clean dùng được.

In [11]:
reloaded = []
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        reloaded.append(json.loads(line))

print("Reloaded records:", len(reloaded))
print("Output file size MB:", round(OUTPUT_PATH.stat().st_size / 1024 / 1024, 3))

for row in reloaded[:3]:
    print("\nID:", row.get("id"))
    print("Source:", row.get("source"), "| Entity:", row.get("entity"), "| Section:", row.get("section"))
    print(row.get("content", "")[:700])


Reloaded records: 218
Output file size MB: 0.654

ID: drug-safety__warfarin__735e158fbd__002__clean_001
Source: MedlinePlus Drug Information | Entity: Warfarin | Section: Before taking warfarin,
có thể bị cắt hoặc chấn thương. Tránh các hoạt động hoặc môn thể thao có nguy cơ bị chấn thương. Gọi cho bác sĩ nếu bạn có tình trạng chảy máu bất thường hoặc nếu bạn ngã và bị thương, đặc biệt là nếu bạn va đầu. Hãy giữ lịch hẹn với bác sĩ và phòng thí nghiệm. Bác sĩ của bạn sẽ chỉ định xét nghiệm máu (PT [test prothrombin] được báo cáo dưới dạng giá trị INR [international normalized ratio]) định kỳ để kiểm tra phản ứng của cơ thể với warfarin. Nếu bác sĩ bảo bạn ngừng dùng warfarin, tác dụng của thuốc có thể kéo dài từ 2 đến 5 ngày sau khi bạn ngừng dùng. Bác sĩ hoặc dược sĩ sẽ cung cấp cho bạn Tài liệu Hướng dẫn Thuốc. Đọc kỹ thông tin và hỏi bác sĩ hoặc dược sĩ nếu bạn có bất kỳ câu hỏi 

ID: drug-safety__warfarin__735e158fbd__002__clean_002
Source: MedlinePlus Drug Information | Entity: Wa